In [ ]:
import argparse
import logging
import numpy as np
import torch
import networkx as nx
from time import time

# Assume these modules exist, import according to actual paths
import model.EIGNN
import preprocessing.importance_calculate as pre_import

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Default configuration constants
DEFAULT_FOLD = 'database/'
DEFAULT_FILENAME = 'cora_graph.npy'
DEFAULT_LENGTH = None  # Needs to be defined according to actual situation
DEFAULT_HIDDEN = 16
DEFAULT_TRAIN_RATIO = 0.5
DEFAULT_EPOCHS = 500
DEFAULT_LR = 0.001
DEFAULT_WEIGHT_DECAY = 5e-5
DEFAULT_SEED = 2025
DEFAULT_TEST_TYPE = 'eignn'   
DEFAULT_MODELS = ['GCN', 'GraphSAGE', 'GIN', 'GCN2', 'SGC']


def parse_args():
    """Parse command line arguments"""
    parser = argparse.ArgumentParser(description='Run GNN experiments with importance calculation')
    parser.add_argument('--fold', type=str, default=DEFAULT_FOLD, help='Data folder')
    parser.add_argument('--filename', type=str, default=DEFAULT_FILENAME, help='Dataset filename')
    parser.add_argument('--length', type=int, default=DEFAULT_LENGTH, help='Length parameter for importance calculation')
    parser.add_argument('--hidden', type=int, default=DEFAULT_HIDDEN, help='Hidden dimension')
    parser.add_argument('--train_ratio', type=float, default=DEFAULT_TRAIN_RATIO, help='Training ratio')
    parser.add_argument('--epochs', type=int, default=DEFAULT_EPOCHS, help='Number of epochs')
    parser.add_argument('--lr', type=float, default=DEFAULT_LR, help='Learning rate')
    parser.add_argument('--weight_decay', type=float, default=DEFAULT_WEIGHT_DECAY, help='Weight decay')
    parser.add_argument('--seed', type=int, default=DEFAULT_SEED, help='Random seed')
    parser.add_argument('--test_type', type=str, choices=['eignn'], default=DEFAULT_TEST_TYPE,
                        help='Type of experiment: eignn')
    parser.add_argument('--models', nargs='+', default=DEFAULT_MODELS,
                        help='List of model names to run')
    return parser.parse_args()


def set_seed(seed: int):
    """Set random seed for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_dataset(fold: str, filename: str):
    """Load raw dataset"""
    path = fold + filename
    logger.info(f"Loading dataset from {path}")
    try:
        dataset = np.load(path, allow_pickle=True).item()
        return dataset
    except Exception as e:
        logger.error(f"Failed to load dataset: {e}")
        raise


def compute_importance(dataset, length):
    """Compute edge and node importance and update dataset"""
    edge_index = dataset['edge_index'].T.tolist()
    edge_list = [tuple(e) for e in edge_index if e[0] != e[1]]
    G = nx.Graph()
    G.add_edges_from(edge_list)

    logger.info("Computing node and edge importance using lambda2...")
    edge_importance, node_importance = pre_import.lamda2_caculate(G, edge_list, lenth=length)

    dataset['edge_importance'] = edge_importance
    dataset['node_importance'] = node_importance
    return dataset


def save_dataset(fold: str, filename: str, dataset):
    """Save updated dataset"""
    path = fold + filename
    logger.info(f"Saving updated dataset to {path}")
    np.save(path, dataset)


def run_model_experiment(dataset, args_dict, test_type: str, model_name: str):
    """Run experiment for a single model, return test accuracy and other metrics"""
    logger.info(f"Running experiment: test_type={test_type}, model={model_name}")
    test_acc, kept_mean, kept_median, removed_mean, removed_median = model.EIGNN.run_experiment(
            dataset, args_dict, conv_type=model_name)
    return test_acc, kept_mean, kept_median, removed_mean, removed_median



def main():
    args = parse_args()

    # Set random seed
    set_seed(args.seed)

    # Prepare device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    logger.info(f"Using device: {device}")

    # Prepare argument dict (to be passed to model experiment function)
    args_dict = {
        'hidden': args.hidden,
        'train_ratio': args.train_ratio,
        'epoch': args.epochs,
        'lr': args.lr,
        'weight_decay': args.weight_decay,
        'seed': args.seed
    }

    # Load raw dataset and compute importance if needed
    dataset = load_dataset(args.fold, args.filename)
    if 'edge_importance' not in dataset or 'node_importance' not in dataset:
        logger.info("Importance not found in dataset, computing...")
        dataset = compute_importance(dataset, args.length)
        save_dataset(args.fold, args.filename, dataset)
    else:
        logger.info("Importance already exists in dataset, skipping computation.")

    # Store results
    results = {args.filename + args.filename: {}}
    eff = {args.filename + args.filename: {}}

    # Run experiment for each model
    for model_name in args.models:
        logger.info(f"\n--- Dataset: {args.filename}, Model: {model_name} ---")
        try:
            test_acc, kept_mean, kept_median, removed_mean, removed_median = run_model_experiment(
                dataset, args_dict, args.test_type, model_name
            )
            results[args.filename + args.filename][model_name] = test_acc
            eff[args.filename + args.filename][model_name] = {
                'kept_mean': kept_mean,
                'kept_median': kept_median,
                'removed_mean': removed_mean,
                'removed_median': removed_median
            }
            logger.info(f"Test accuracy: {test_acc:.4f}, Kept mean: {kept_mean:.4f}, Removed mean: {removed_mean:.4f}")

    # Optionally save results and eff to file or further process
    logger.info("All experiments finished.")


if __name__ == "__main__":
    main()